In [1]:
import subprocess
subprocess.run(["py", "-m", "pip", "install", "earthengine-api"], capture_output=True)
print(" library installed")

 library installed


In [1]:
import subprocess
subprocess.run(["py", "-m", "pip", "install", "pystac-client", "planetary-computer", "stackstac", "rioxarray"], 
               capture_output=True)
print("✓ Libraries installed")

✓ Libraries installed


In [2]:
import planetary_computer
import pystac_client
import stackstac
import numpy as np
import pandas as pd
from datetime import datetime

# Connect to Microsoft Planetary Computer (no login needed)
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Uasin Gishu bounding box
bbox = [34.9, 0.1, 35.7, 1.1]

print("✓ Connected to Microsoft Planetary Computer")
print("Testing search for MODIS NDVI data...")

# Test search for one year first
search = catalog.search(
    collections=["modis-13A3-061"],  # MODIS monthly NDVI
    bbox=bbox,
    datetime="2012-01-01/2012-12-31",
)

items = list(search.items())
print(f"✓ Found {len(items)} MODIS tiles for 2012")
if items:
    print(f"  First item: {items[0].id}")
    print(f"  Available bands: {list(items[0].assets.keys())[:5]}")

✓ Connected to Microsoft Planetary Computer
Testing search for MODIS NDVI data...
✓ Found 0 MODIS tiles for 2012


In [3]:
# Search for available NDVI-related collections
collections = catalog.get_collections()

print("Looking for NDVI/vegetation collections...")
for col in collections:
    if any(k in col.id.lower() for k in ["modis", "ndvi", "vegetation", "mod13"]):
        print(f"  Found: {col.id} — {col.title}")

Looking for NDVI/vegetation collections...
  Found: modis-64A1-061 — MODIS Burned Area Monthly
  Found: modis-17A2H-061 — MODIS Gross Primary Productivity 8-Day
  Found: modis-11A2-061 — MODIS Land Surface Temperature/Emissivity 8-Day
  Found: modis-17A2HGF-061 — MODIS Gross Primary Productivity 8-Day Gap-Filled
  Found: modis-17A3HGF-061 — MODIS Net Primary Production Yearly Gap-Filled
  Found: modis-09A1-061 — MODIS Surface Reflectance 8-Day (500m)
  Found: modis-16A3GF-061 — MODIS Net Evapotranspiration Yearly Gap-Filled
  Found: modis-21A2-061 — MODIS Land Surface Temperature/3-Band Emissivity 8-Day
  Found: modis-43A4-061 — MODIS Nadir BRDF-Adjusted Reflectance (NBAR) Daily
  Found: modis-09Q1-061 — MODIS Surface Reflectance 8-Day (250m)
  Found: modis-14A1-061 — MODIS Thermal Anomalies/Fire Daily
  Found: modis-13Q1-061 — MODIS Vegetation Indices 16-Day (250m)
  Found: modis-14A2-061 — MODIS Thermal Anomalies/Fire 8-Day
  Found: modis-15A2H-061 — MODIS Leaf Area Index/FPAR 8-Day


In [4]:
# Search using correct collection name
search = catalog.search(
    collections=["modis-13Q1-061"],  # MODIS Vegetation Indices 250m
    bbox=bbox,
    datetime="2012-01-01/2012-12-31",
)

items = list(search.items())
print(f"✓ Found {len(items)} tiles for 2012")
if items:
    print(f"  First item: {items[0].id}")
    print(f"  Available bands: {list(items[0].assets.keys())}")

✓ Found 48 tiles for 2012
  First item: MYD13Q1.A2012361.h21v08.061.2021223184756
  Available bands: ['hdf', 'metadata', '250m_16_days_EVI', '250m_16_days_NDVI', '250m_16_days_VI_Quality', '250m_16_days_MIR_reflectance', '250m_16_days_NIR_reflectance', '250m_16_days_red_reflectance', '250m_16_days_blue_reflectance', '250m_16_days_sun_zenith_angle', '250m_16_days_pixel_reliability', '250m_16_days_view_zenith_angle', '250m_16_days_relative_azimuth_angle', '250m_16_days_composite_day_of_the_year', 'tilejson', 'rendered_preview']


In [5]:
import stackstac
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

bbox = [34.9, 0.1, 35.7, 1.1]
results = []

print("Extracting NDVI for Uasin Gishu 2005–2024...")
print("(This will take 3–5 minutes — processing 20 years)\n")

for year in range(2005, 2025):
    try:
        # Long rains season: March–July
        search_lr = catalog.search(
            collections=["modis-13Q1-061"],
            bbox=bbox,
            datetime=f"{year}-03-01/{year}-07-31",
        )
        items_lr = list(search_lr.items())

        # Short rains season: October–December
        search_sr = catalog.search(
            collections=["modis-13Q1-061"],
            bbox=bbox,
            datetime=f"{year}-10-01/{year}-12-31",
        )
        items_sr = list(search_sr.items())

        ndvi_lr = None
        ndvi_sr = None

        if items_lr:
            stack_lr = stackstac.stack(
                items_lr,
                assets=["250m_16_days_NDVI"],
                bounds_latlon=bbox,
                resolution=250,
            )
            # Scale NDVI (MODIS stores as integers, scale = 0.0001)
            ndvi_lr = float(stack_lr.mean().compute()) * 0.0001

        if items_sr:
            stack_sr = stackstac.stack(
                items_sr,
                assets=["250m_16_days_NDVI"],
                bounds_latlon=bbox,
                resolution=250,
            )
            ndvi_sr = float(stack_sr.mean().compute()) * 0.0001

        results.append({
            "Year": year,
            "NDVI_LongRains": round(ndvi_lr, 4) if ndvi_lr else None,
            "NDVI_ShortRains": round(ndvi_sr, 4) if ndvi_sr else None,
        })
        print(f"  ✓ {year} — Long Rains NDVI: {ndvi_lr:.4f} | Short Rains NDVI: {ndvi_sr:.4f}")

    except Exception as e:
        print(f"  ✗ {year} failed: {e}")
        results.append({"Year": year, "NDVI_LongRains": None, "NDVI_ShortRains": None})

# Save
df_ndvi = pd.DataFrame(results)
df_ndvi.to_csv("../data/processed/ndvi_clean.csv", index=False)

print(f"\n✓ Done! Saved ndvi_clean.csv")
print(df_ndvi)

Extracting NDVI for Uasin Gishu 2005–2024...
(This will take 3–5 minutes — processing 20 years)

  ✗ 2005 failed: Cannot pick a common CRS, since asset '250m_16_days_NDVI' of item 0 'MOD13Q1.A2005209.h21v08.061.2020241221814' does not have one.

Please specify a CRS with the `epsg=` argument.
  ✗ 2006 failed: Cannot pick a common CRS, since asset '250m_16_days_NDVI' of item 0 'MOD13Q1.A2006209.h21v08.061.2020269045705' does not have one.

Please specify a CRS with the `epsg=` argument.
  ✗ 2007 failed: Cannot pick a common CRS, since asset '250m_16_days_NDVI' of item 0 'MOD13Q1.A2007209.h21v08.061.2021071054223' does not have one.

Please specify a CRS with the `epsg=` argument.
  ✗ 2008 failed: Cannot pick a common CRS, since asset '250m_16_days_NDVI' of item 0 'MOD13Q1.A2008209.h21v08.061.2021104100608' does not have one.

Please specify a CRS with the `epsg=` argument.
  ✗ 2009 failed: Cannot pick a common CRS, since asset '250m_16_days_NDVI' of item 0 'MOD13Q1.A2009209.h21v08.061.2

In [8]:
import requests

sites = [
    ("Google", "https://www.google.com"),
    ("NASA POWER", "https://power.larc.nasa.gov"),
    ("ORNL MODIS", "https://modis.ornl.gov"),
    ("Planetary Computer", "https://planetarycomputer.microsoft.com"),
]

print("Testing connections...\n")
for name, url in sites:
    try:
        r = requests.get(url, timeout=10)
        print(f"  ✓ {name}: reachable (status {r.status_code})")
    except Exception as e:
        print(f"  ✗ {name}: FAILED — {e}")

Testing connections...

  ✓ Google: reachable (status 200)
  ✓ NASA POWER: reachable (status 200)
  ✓ ORNL MODIS: reachable (status 200)
  ✓ Planetary Computer: reachable (status 200)


In [9]:
import requests
import pandas as pd
import numpy as np

LAT = 0.5204
LON = 35.2699

# Test with just 2012 first
print("Testing ORNL MODIS for 2012...")

url = (
    f"https://modis.ornl.gov/rst/api/v1/MOD13Q1/subset"
    f"?latitude={LAT}&longitude={LON}"
    f"&startDate=A2012060"
    f"&endDate=A2012212"
    f"&kmAboveBelow=0&kmLeftRight=0"
)

try:
    r = requests.get(url, timeout=60, headers={"Accept":"application/json"})
    print(f"Status: {r.status_code}")
    data = r.json()
    print(f"Keys returned: {list(data.keys())}")
    
    # Show first subset item
    if "subset" in data:
        print(f"Number of records: {len(data['subset'])}")
        print(f"First record: {data['subset'][0]}")
    else:
        print(f"Response: {str(data)[:300]}")
except Exception as e:
    print(f"Failed: {e}")

Testing ORNL MODIS for 2012...
Status: 200
Keys returned: ['xllcorner', 'yllcorner', 'cellsize', 'nrows', 'ncols', 'band', 'latitude', 'longitude', 'header', 'subset']
Number of records: 120
First record: {'modis_date': 'A2012065', 'calendar_date': '2012-03-05', 'band': '250m_16_days_blue_reflectance', 'tile': 'h21v08', 'proc_date': '2021205122218', 'data': [509]}


In [10]:
import requests
import pandas as pd
import numpy as np
import time

LAT = 0.5204
LON = 35.2699
results = []

print("Extracting NDVI for Uasin Gishu 2005–2024...\n")

for year in range(2005, 2025):
    try:
        # Long rains: March–July (day 060–212)
        url_lr = (
            f"https://modis.ornl.gov/rst/api/v1/MOD13Q1/subset"
            f"?latitude={LAT}&longitude={LON}"
            f"&startDate=A{year}060"
            f"&endDate=A{year}212"
            f"&kmAboveBelow=0&kmLeftRight=0"
        )
        r_lr = requests.get(url_lr, timeout=60,
                            headers={"Accept":"application/json"}).json()

        # Short rains: October–December (day 273–365)
        url_sr = (
            f"https://modis.ornl.gov/rst/api/v1/MOD13Q1/subset"
            f"?latitude={LAT}&longitude={LON}"
            f"&startDate=A{year}273"
            f"&endDate=A{year}365"
            f"&kmAboveBelow=0&kmLeftRight=0"
        )
        r_sr = requests.get(url_sr, timeout=60,
                            headers={"Accept":"application/json"}).json()

        # Filter for NDVI band only and valid values
        ndvi_lr_vals = [
            s["data"][0] for s in r_lr.get("subset", [])
            if s["band"] == "250m_16_days_NDVI" and s["data"][0] > -2000
        ]
        ndvi_sr_vals = [
            s["data"][0] for s in r_sr.get("subset", [])
            if s["band"] == "250m_16_days_NDVI" and s["data"][0] > -2000
        ]

        ndvi_lr = round(np.mean(ndvi_lr_vals) * 0.0001, 4) if ndvi_lr_vals else None
        ndvi_sr = round(np.mean(ndvi_sr_vals) * 0.0001, 4) if ndvi_sr_vals else None

        results.append({
            "Year": year,
            "NDVI_LongRains":  ndvi_lr,
            "NDVI_ShortRains": ndvi_sr
        })
        print(f"  ✓ {year} — Long Rains: {ndvi_lr} | Short Rains: {ndvi_sr}")
        time.sleep(1)  # be polite to the server

    except Exception as e:
        print(f"  ✗ {year} failed: {e}")
        results.append({"Year": year, "NDVI_LongRains": None, "NDVI_ShortRains": None})
        time.sleep(2)  # wait longer after a failure

df_ndvi = pd.DataFrame(results)
df_ndvi.to_csv("../data/processed/ndvi_clean.csv", index=False)
print(f"\n✓ Saved ndvi_clean.csv")
print(df_ndvi)

Extracting NDVI for Uasin Gishu 2005–2024...

  ✓ 2005 — Long Rains: 0.2555 | Short Rains: 0.184
  ✓ 2006 — Long Rains: 0.2456 | Short Rains: 0.2754
  ✓ 2007 — Long Rains: 0.2932 | Short Rains: 0.1968
  ✓ 2008 — Long Rains: 0.251 | Short Rains: 0.2579
  ✓ 2009 — Long Rains: 0.2328 | Short Rains: 0.23
  ✓ 2010 — Long Rains: 0.2561 | Short Rains: 0.2422
  ✓ 2011 — Long Rains: 0.2585 | Short Rains: 0.2642
  ✓ 2012 — Long Rains: 0.2598 | Short Rains: 0.2619
  ✓ 2013 — Long Rains: 0.2724 | Short Rains: 0.2534
  ✓ 2014 — Long Rains: 0.2659 | Short Rains: 0.2437
  ✓ 2015 — Long Rains: 0.264 | Short Rains: 0.2562
  ✓ 2016 — Long Rains: 0.2394 | Short Rains: 0.2178
  ✓ 2017 — Long Rains: 0.2699 | Short Rains: 0.2447
  ✓ 2018 — Long Rains: 0.2821 | Short Rains: 0.22
  ✓ 2019 — Long Rains: 0.2303 | Short Rains: 0.29
  ✓ 2020 — Long Rains: 0.269 | Short Rains: 0.2675
  ✓ 2021 — Long Rains: 0.2371 | Short Rains: 0.2222
  ✓ 2022 — Long Rains: 0.2518 | Short Rains: 0.231
  ✓ 2023 — Long Rains: 0.2484

what the NDVI values mean:

0.2–0.3 — moderate vegetation (typical dry season)
0.3–0.5 — good crop cover (healthy growing season)
0.5+ — dense healthy vegetation (excellent crop conditions)



2007 had the best Long Rains NDVI (0.2932) — expect good yield that year
2016 was the worst Long Rains (0.2394) — likely a poor harvest
2019 had the best Short Rains (0.29) — good late season recovery
Values are consistently in the 0.23–0.29 range — typical for highland Kenya mixed farmland